# PicoCal - Per-region performance incl R4 (notebook 11)

Mentor item: train on all regions with a **global split** (test held out from every region), then break the test resolution down **per region** to see how the model does on R4 (very different statistics). kNN-25, energy cut 1-100 GeV.

In [1]:
import sys, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, select_knn, split, resolution, TokenDS, collate, EPS

SEEDS = 3
cfg = {"d": 96, "nhead": 4, "layers": 3, "dropout": 0.1, "lr": 3e-4, "wd": 1e-4,
       "batch": 128, "epochs": 150, "patience": 25}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
files = sorted((repo / "data" / "full").glob("matched_*.root"))[:100]
RN = ["R0_15mm", "R1_30mm", "R2_40mm", "R3_60mm", "R4_120mm"]
{"device": DEVICE, "files": len(files), "seeds": SEEDS}

{'device': 'cuda', 'files': 100, 'seeds': 3}

In [2]:
class TunedTransformer(nn.Module):
    def __init__(self, in_dim, d=96, nhead=4, layers=3, dropout=0.1):
        super().__init__()
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=4 * d, dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, 1))

    def forward(self, x, m):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = m.unsqueeze(-1).float()
        return self.head((h * w).sum(1) / w.sum(1).clamp(min=1))


def train_model(toks, y, atr, ava, in_dim, seed):
    torch.manual_seed(seed)
    cont = np.concatenate([toks[i][:, :7] for i in atr], 0)
    mean = cont.mean(0); std = cont.std(0) + EPS

    def loader(idx, sh, bs):
        return DataLoader(TokenDS([toks[i] for i in idx], y[idx], mean, std, 7),
                          batch_size=bs, shuffle=sh, collate_fn=collate)

    model = TunedTransformer(in_dim, cfg["d"], cfg["nhead"], cfg["layers"], cfg["dropout"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    dl_tr, dl_va = loader(atr, True, cfg["batch"]), loader(ava, False, 256)

    def vloss():
        model.eval(); s = 0.0; c = 0
        with torch.no_grad():
            for X, m, yb in dl_va:
                s += nn.functional.mse_loss(model(X.to(DEVICE), m.to(DEVICE)), yb.to(DEVICE)).item(); c += 1
        return s / max(c, 1)

    best = 1e9; bstate = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train()
        for X, m, yb in dl_tr:
            opt.zero_grad()
            nn.functional.mse_loss(model(X.to(DEVICE), m.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4:
            best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break
    model.load_state_dict(bstate); model.eval()
    model._mean, model._std = mean, std
    return model


def predict(model, toks, y, idx):
    dl = DataLoader(TokenDS([toks[i] for i in idx], y[idx], model._mean, model._std, 7),
                    batch_size=256, shuffle=False, collate_fn=collate)
    out = []
    with torch.no_grad():
        for X, m, _ in dl:
            out.append(model(X.to(DEVICE), m.to(DEVICE)).cpu().numpy().ravel())
    return np.concatenate(out)

In [3]:
D = build(files, 3, 100.0, selector=lambda c: select_knn(c, 25))
toks = D["tok_seed"]; y = D["y"]; Et = D["Etrue"]; region = D["region"]
keep = (Et >= 1.0) & (Et <= 100.0); kept = np.flatnonzero(keep)
a_, v_, t_ = split(len(kept)); atr, ava, ate = kept[a_], kept[v_], kept[t_]
in_dim = toks[int(kept[0])].shape[1]
print("train", len(atr), "test", len(ate), "| per-region test:",
      {RN[rg]: int((region[ate] == rg).sum()) for rg in range(5)})

per_seed = {rg: [] for rg in range(5)}
for s in range(SEEDS):
    model = train_model(toks, y, atr, ava, in_dim, s)
    pv = predict(model, toks, y, ava); ca, cb = np.polyfit(pv, y[ava], 1)
    for rg in range(5):
        sel = ate[region[ate] == rg]
        if len(sel) >= 20:
            per_seed[rg].append(resolution(np.exp(ca * predict(model, toks, y, sel) + cb), Et[sel])["sigma_eff"])
    print(f"seed {s} done", flush=True)

train 23431 test 5021 | per-region test: {'R0_15mm': 589, 'R1_30mm': 1097, 'R2_40mm': 1350, 'R3_60mm': 1706, 'R4_120mm': 279}


seed 0 done


seed 1 done


seed 2 done


In [4]:
ta, tb = np.polyfit(np.log(D["total_energy"][atr] + EPS), y[atr], 1)
gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(D["agg"][atr], y[atr])

rows = []
for rg in range(5):
    sel = ate[region[ate] == rg]
    if len(sel) < 20:
        rows.append({"region": RN[rg], "n_test": int(len(sel)), "transformer": None,
                     "tf_std": None, "BDT": None, "total_energy": None}); continue
    tf = np.array(per_seed[rg])
    te = resolution(np.exp(ta * np.log(D["total_energy"][sel] + EPS) + tb), Et[sel])["sigma_eff"]
    bd = resolution(np.exp(gb.predict(D["agg"][sel])), Et[sel])["sigma_eff"]
    rows.append({"region": RN[rg], "n_test": int(len(sel)),
                 "transformer": round(float(tf.mean()), 4), "tf_std": round(float(tf.std()), 4),
                 "BDT": round(float(bd), 4), "total_energy": round(float(te), 4)})
per_region = pd.DataFrame(rows)
per_region

,region,n_test,transformer,tf_std,BDT,total_energy
0,R0_15mm,589,0.0838,0.0089,0.0628,0.0731
1,R1_30mm,1097,0.0706,0.0128,0.0671,0.0717
2,R2_40mm,1350,0.0484,0.0038,0.0431,0.0680
3,R3_60mm,1706,0.0513,0.0071,0.0412,0.0662
4,R4_120mm,279,0.0612,0.0063,0.0412,0.0570


In [5]:
import plotly.graph_objects as go
d = per_region.dropna()
fig = go.Figure()
fig.add_trace(go.Bar(name="transformer", x=d["region"], y=d["transformer"],
                     error_y=dict(type="data", array=d["tf_std"]), marker_color="#2ca02c"))
fig.add_trace(go.Bar(name="BDT", x=d["region"], y=d["BDT"], marker_color="#8c8c8c"))
fig.add_trace(go.Bar(name="total_energy", x=d["region"], y=d["total_energy"], marker_color="#d62728"))
fig.update_layout(barmode="group", template="plotly_white", height=440,
                  title="Per-region resolution (kNN-25, all-region training, global split)",
                  yaxis_title="sigma_eff", legend_title="")
fig.show()

Read across regions: if R4 (120 mm, few clusters, ~9 cells) has clearly the worst sigma_eff, that matches Felipe's concern that the all-region model transfers poorly to R4's very different statistics. Where the transformer beats `total_energy` / BDT per region tells us where the learned model actually helps.